# FIRST_VALUE, LAST_VALUE, and NTH_VALUE

## Overview
These functions retrieve the value of an expression from a specific row within the window frame — the first, last, or nth row.

```sql
FIRST_VALUE(expression) OVER (PARTITION BY ... ORDER BY ... [frame_clause])
LAST_VALUE(expression)  OVER (PARTITION BY ... ORDER BY ... [frame_clause])
NTH_VALUE(expression, n) OVER (PARTITION BY ... ORDER BY ... [frame_clause])
```

**Critical frame behaviour:**

| Function | Default frame | Consequence |
|---|---|---|
| `FIRST_VALUE` | `RANGE UNBOUNDED PRECEDING TO CURRENT ROW` | Works as expected — first row of partition is stable |
| `LAST_VALUE` | `RANGE UNBOUNDED PRECEDING TO CURRENT ROW` | **Returns the current row's value**, not the last in partition! |
| `NTH_VALUE` | `RANGE UNBOUNDED PRECEDING TO CURRENT ROW` | Returns NULL if the nth row hasn't been reached yet |

**The LAST_VALUE trap** is the most common window function bug. Because the default frame ends at `CURRENT ROW`, `LAST_VALUE` always returns the value of the current row (the last row seen so far). To get the actual last value in the partition, you must explicitly extend the frame: `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, account_id INTEGER, trade_date TEXT, ticker TEXT, direction TEXT, shares INTEGER, price REAL, total_value REAL);
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,101,'2023-03-05','Deposit',4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',15000.00,'Payroll',0),(1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',15000.00,'Payroll',0),(1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',15000.00,'Payroll',0),(1012,105,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1014,105,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),(1016,107,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',-25.00,'Fee',1),(1018,107,'2023-03-15','Withdrawal',-450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',2800.00,'Payroll',0),(1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',4500.00,'Payroll',0),(1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',4500.00,'Payroll',0);
INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy',10,148.50,1485.00),(2002,104,'2023-01-25','MSFT','Buy',5,240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy',5,153.20,766.00),(2004,104,'2023-02-28','MSFT','Sell',3,252.80,758.40),
  (2005,104,'2023-03-15','NVDA','Buy',2,278.50,557.00),(2006,108,'2023-01-05','AAPL','Buy',20,130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy',8,95.20,761.60),(2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy',5,99.80,499.00),(2010,108,'2023-03-10','NVDA','Buy',4,265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy',8,238.00,1904.00),(2012,110,'2023-02-01','AAPL','Buy',15,145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy',3,248.50,745.50),(2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy',6,280.00,1680.00);
""")
conn.commit()
print('Schema ready.')


---
## FIRST_VALUE — stable across the partition

In [ ]:
# First transaction amount and date for each account
print('=== FIRST_VALUE: first transaction details per account ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        FIRST_VALUE(amount)   OVER w  AS first_txn_amount,
        FIRST_VALUE(txn_date) OVER w  AS first_txn_date,
        ROUND(amount - FIRST_VALUE(amount) OVER w, 2) AS change_from_first
FROM    transactions
WHERE   account_id IN (101, 103)
WINDOW  w AS (PARTITION BY account_id ORDER BY txn_date, txn_id
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
ORDER BY account_id, txn_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('FIRST_VALUE is the same for every row in the partition — the opener.')


---
## LAST_VALUE — the frame trap

In [ ]:
# Demonstrate the LAST_VALUE frame trap
print('=== LAST_VALUE trap: default frame returns current row, not last in partition ===')
q_wrong = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        LAST_VALUE(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            -- Default: RANGE UNBOUNDED PRECEDING TO CURRENT ROW
        )  AS last_val_default   -- WRONG: returns current row's amount
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date, txn_id
"""
print('With default frame (WRONG — returns current row):')
print(pd.read_sql(q_wrong, conn).to_string(index=False))

print()
q_correct = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        LAST_VALUE(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )  AS last_val_correct   -- CORRECT: extends frame to end of partition
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date, txn_id
"""
print('With explicit full frame (CORRECT — same value for all rows in partition):')
print(pd.read_sql(q_correct, conn).to_string(index=False))
print()
print('Fix: always use ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING for LAST_VALUE.')


---
## NTH_VALUE — accessing a specific position

In [ ]:
# 2nd highest-value trade per account and ticker
print('=== NTH_VALUE: 2nd trade in sequence per account ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        NTH_VALUE(amount, 1) OVER w  AS first_amount,
        NTH_VALUE(amount, 2) OVER w  AS second_amount,
        NTH_VALUE(amount, 3) OVER w  AS third_amount
FROM    transactions
WHERE   account_id = 101
WINDOW  w AS (
    PARTITION BY account_id
    ORDER BY txn_date, txn_id
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
)
ORDER BY txn_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('NTH_VALUE returns NULL until the nth row exists in the frame.')
print('With UNBOUNDED FOLLOWING, all rows see the full partition.')


---
## Practical: first buy price vs current price per ticker

In [ ]:
# Compare each trade price to the first buy price for that ticker
# and the most recent (last) price
print('=== Trade vs first and last price per ticker ===')
q = """
SELECT  trade_id,
        account_id,
        trade_date,
        ticker,
        price,
        FIRST_VALUE(price) OVER (
            PARTITION BY ticker
            ORDER BY trade_date, trade_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )                                                    AS first_price,
        LAST_VALUE(price) OVER (
            PARTITION BY ticker
            ORDER BY trade_date, trade_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )                                                    AS last_price,
        ROUND(price - FIRST_VALUE(price) OVER (
            PARTITION BY ticker ORDER BY trade_date, trade_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS chg_from_first,
        ROUND(100.0 * (price - FIRST_VALUE(price) OVER (
            PARTITION BY ticker ORDER BY trade_date, trade_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW))
            / FIRST_VALUE(price) OVER (
            PARTITION BY ticker ORDER BY trade_date, trade_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 1) AS pct_chg_from_first
FROM    trades
ORDER BY ticker, trade_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. LAST_VALUE with default frame always returns the current row's value**
This is the single most common window function bug. `LAST_VALUE(col) OVER (ORDER BY date)` uses the default frame `RANGE UNBOUNDED PRECEDING TO CURRENT ROW`, so the "last" row is always the current row. Always specify `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` when you want the true last value in the partition.

**2. NTH_VALUE returns NULL until the nth row is in the frame**
`NTH_VALUE(amount, 3)` returns NULL for the first two rows of every partition when using the default frame. With `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`, all rows in the partition see the 3rd value (or NULL if the partition has fewer than 3 rows).

**3. FIRST_VALUE is stable but LAST_VALUE requires an explicit frame**
A common pattern is to use the same WINDOW definition for FIRST_VALUE and LAST_VALUE — but the default frame makes LAST_VALUE wrong. Either use two separate OVER clauses or accept that you need `UNBOUNDED FOLLOWING` for LAST_VALUE specifically.

**4. NTH_VALUE n=0 or n > partition size**
`NTH_VALUE(amount, 0)` is an error (1-indexed). `NTH_VALUE(amount, 100)` in a 5-row partition returns NULL. Always validate that your n value is within the expected partition size range, or COALESCE the result.

**5. Performance: UNBOUNDED FOLLOWING forces a full partition read**
Adding `UNBOUNDED FOLLOWING` to the frame means the database must read all rows in the partition before processing any row. For very large partitions, this can be memory-intensive. For FIRST_VALUE, the default frame is fine and more efficient — only use the extended frame where needed.

**6. Confusing row-level FIRST/LAST with aggregate MIN/MAX**
`FIRST_VALUE` returns the value from the first row by ORDER BY, which may not be the minimum. `MIN(amount) OVER (PARTITION BY account_id)` returns the smallest amount across the partition, regardless of row order. These are different operations — choose based on whether you need "earliest" or "smallest."


---
*sql_methods_library - Samantha McGarrigle*